# Modules tour: Predict through Refine

This notebook walks through the core DSPy modules covered in Chapter 7, sections 7.1.1 to 7.1.5: `Predict`, `ChainOfThought`, `MultiChainComparison`, `BestOfN`, and `Refine`. Each module wraps the same Signature in a different inference strategy.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## 7.1.1 Predict

The foundational module. Every other module delegates to Predict under the hood. You can override configuration per call using the `config` parameter.


In [ ]:
classify = dspy.Predict("review: str -> sentiment: str")

# Default call
result = classify(review="Great product, love it!")

# Override temperature for this specific call
result = classify(review="It was okay I guess", config={"temperature": 0.9})


## 7.1.2 ChainOfThought with a custom rationale field

`ChainOfThought` prepends a reasoning field to your Signature. You can customize the reasoning field if the default prefix doesn't fit your task.


In [ ]:
# Custom rationale field for medical triage
custom_rationale = dspy.OutputField(
    prefix="Clinical reasoning: Based on the symptoms, I should consider"
)

triage = dspy.ChainOfThought(
    "symptoms: str -> urgency: str",
    rationale_field=custom_rationale
)

result = triage(symptoms="Patient reports sudden severe headache and stiff neck")
print(result.reasoning)  # Uses your custom prefix
print(result.urgency)


## 7.1.3 MultiChainComparison

Generate M reasoning attempts and let the model synthesize the best answer from them.


In [ ]:
cot = dspy.ChainOfThought("clause: str -> interpretation: str")

# Generate 3 reasoning attempts
result = cot(
    clause="The licensee may sublicense the software to affiliates only",
    config={"n": 3, "temperature": 0.9}
)

# Let the model compare all 3 reasoning paths
compare = dspy.MultiChainComparison("clause: str -> interpretation: str", M=3)
final = compare(clause=result.clause, completions=result.completions)
print(final.interpretation)


## 7.1.4 BestOfN with a haiku + syllable reward

`BestOfN` retries with a reward function and supports early stopping via the `threshold` parameter.


In [ ]:
haiku_writer = dspy.ChainOfThought("topic: str -> haiku: str")

def syllable_check(args, pred):
    """Check if the haiku follows 5-7-5 syllable structure."""
    lines = pred.haiku.strip().split("\n")
    if len(lines) != 3:
        return 0.0
    # Rough syllable counting (production code would use a library)
    counts = [len([c for c in line.split() if c]) for line in lines]
    return 1.0 if counts == [5, 7, 5] else 0.3

best_haiku = dspy.BestOfN(
    module=haiku_writer,
    N=5,
    reward_fn=syllable_check,
    threshold=1.0  # Stop as soon as we get a perfect haiku
)

result = best_haiku(topic="autumn leaves falling")


## 7.1.5 Refine

Same API as `BestOfN`, but when an attempt fails the threshold, Refine generates targeted feedback for the next retry using an internal `OfferFeedback` Signature.


In [ ]:
sql_writer = dspy.ChainOfThought(
    "question: str, db_schema: str -> sql_query: str"
)

def validate_sql(args, pred):
    """Validate the SQL query is syntactically correct and answers the question."""
    query = pred.sql_query.strip()
    if not query.upper().startswith("SELECT"):
        return 0.0
    if args["db_schema"].split(",")[0].strip() not in query:
        return 0.3
    return 1.0

refined_sql = dspy.Refine(
    module=sql_writer,
    N=4,
    reward_fn=validate_sql,
    threshold=1.0
)

result = refined_sql(
    question="How many orders were placed last month?",
    db_schema="orders (id, customer_id, placed_at, total), customers (id, name)"
)
